In [1]:
# Get raw data
with open('input/19.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
from intcode import Program  # From Day 9
import numpy as np

In [3]:
# Part 1
class Drone(Program):
    instr = [int(i) for i in rawinput.split(',')]
    def __init__(self, x, y, verbose=False):
        super().__init__(self.instr, [x, y], verbose)
        
    @classmethod
    def assess(cls, x, y, verbose=False):
        obj = cls(x, y, verbose)
        obj.do_exec()
        return obj.output[0]
    
scan = np.zeros((50,50), dtype=int)
for y in range(50):
    for x in range(50):
        if Drone.assess(x,y):
            scan[y,x] = 1

np.sum(scan).item()

181

In [4]:
# Part 2
from sklearn.linear_model import LinearRegression

topix = np.arange(scan.size)[((scan == 0) 
                              & (np.pad(scan[1:], ((0,1),(0,0))) == 1)
                              & (np.pad(scan[:, :-1], ((0,0),(1,0))) == 1)).reshape(-1)]
btmix = np.arange(scan.size)[((scan == 0) 
                              & (np.pad(scan[:-1], ((1,0),(0,0))) == 1)
                              & (np.pad(scan[:, 1:], ((0,0),(0,1))) == 1)).reshape(-1)]
top_c,top_i,btm_c,btm_i = [np.squeeze(getattr(reg,j)).item()
                           for i in [topix,btmix]
                           for reg in[LinearRegression().fit(np.expand_dims(i % scan.shape[1], 1), 
                                                             i // scan.shape[1])]
                           for j in['coef_','intercept_']]

def sqfit(size):
    y = 0
    while (y-top_i)/top_c - (y-btm_i+size)/btm_c < size:  y+=1
    return y

yl, yh = sqfit(95), (sfh:=sqfit(105))+105
xl, xh = int((yl+95-btm_i)/btm_c), int((sfh-top_i)/top_c)+1

scan2 = np.zeros(((yr:=yh-yl),(xr:=xh-xl)), dtype=int)
for y in range(yr):
    for x in range(xr):
        if Drone.assess(x+xl,y+yl):
            scan2[y,x] = 1
            
def closest_square(scan2, size):
    csums = np.pad(np.cumsum(np.cumsum(scan2[::-1, ::-1], axis=0), axis=1)[::-1, ::-1],
                   ((0,1),(0,1)))
    csum_diffs = (csums[:-size, :-size] + csums[size:, size:] 
                  - csums[size:, :-size] - csums[:-size, size:])
    sq = np.column_stack(((y:=np.arange(csum_diffs.size)
                           [csum_diffs.reshape(-1) == size ** 2]) 
                          // (z:=csum_diffs.shape[1]), 
                          y%z))
    return sq[np.argsort(np.sum(sq ** 2, axis=1))][0] if sq.shape[0] else None
           
np.sum((closest_square(scan2, 100)+[yl,xl]) * [1,10000]).item()

4240964